<a href="https://colab.research.google.com/github/littleShaniZ/ControlNet-v1-1-nightly/blob/home_assignment/notebooks/SelectiveControlNet_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Selective ControlNet - Adapted from v1.1 Lineart Script

Install required packages

In [ ]:
!pip install -q transformers diffusers xformers accelerate
!pip install -q git+https://github.com/fcakyon/u2net

Clone ControlNet repo and move into it

In [ ]:
!git clone https://github.com/lllyasviel/ControlNet-v1-1-nightly.git
%cd ControlNet-v1-1-nightly

Import libraries

In [ ]:
import torch
from PIL import Image
import requests
import cv2
import numpy as np
from io import BytesIO
import matplotlib.pyplot as plt
from torchvision import transforms
from u2net.infer import main as u2net_infer

from cldm.model import create_model, load_state_dict
from cldm.ddim_hacked import DDIMSampler
from annotator.lineart import apply_lineart

Setup model

In [ ]:
model = create_model('./configs/controlnet.yaml').cpu()
model.load_state_dict(load_state_dict('./models/control_sd15_lineart.pth', location='cuda'))
model = model.cuda()
model.eval()
sampler = DDIMSampler(model)

Load and preprocess input image

In [ ]:
image_url = "https://raw.githubusercontent.com/lllyasviel/ControlNet/main/images/bird.png"
response = requests.get(image_url)
input_image = Image.open(BytesIO(response.content)).convert("RGB").resize((512, 512))
input_np = np.array(input_image)

Prepering the input to the model

In [ ]:
# Generate lineart hint
hint_np = apply_lineart(input_np)

# Generate foreground mask using U2Net
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])
image_tensor = transform(input_image).unsqueeze(0)
mask = u2net_infer(image_tensor.squeeze(0))
mask = mask.resize((512, 512))
mask_np = (np.array(mask) > 127).astype(np.uint8)

# Apply mask to lineart
masked_hint = hint_np.copy()
masked_hint[mask_np == 0] = 255  # white out background in hint

# Define prompts
prompt = "a bird with armor, fantasy style"
n_prompt = "blurry, distorted, low quality"

# Cell 10: Prepare ControlNet conditioning
cond = torch.tensor(masked_hint / 255.0).float()
cond = cond[None, None, :, :].repeat(1, 3, 1, 1).cuda()

# Sampling configuration
ddim_steps = 30
guess_mode = False
strength = 1.0
guide_scale = 9.0
eta = 0.0